In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (HyPepTox-Fuse)

This notebook curates the **HyPepTox-Fuse** dataset from FASTA files. Peptide sequences are parsed from multiple inputs, binary toxicity labels are inferred directly from FASTA record identifiers, duplicate consistency checks are applied, and the final curated dataset and metadata are exported for downstream analysis.

- **Toxic effect / endpoint:** toxic
- **Source:** HyPepTox-Fuse
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads FASTA-like files** (`.fasta`, `.fa`, `.faa`, `.txt`) from the HyPepTox-Fuse input directory.
- **Infers toxicity labels from FASTA record IDs**:
  - if `id` contains `"Positive"` (case-insensitive) → `label = 1`,
  - if `id` contains `"Negative"` (case-insensitive) → `label = 0`,
  - otherwise `label = NA` (missing/unknown).
- **Keeps a standardized schema**:
  - `sequence`
  - `label`
- **Checks duplicated sequences** by sequence:
  - unique sequences are retained,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels are flagged as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_toxic_dataset.csv`,
  - `detected_error_sequences.csv`,
  - `metadata.json`.

In [2]:
name_source = "HyPepTox-Fuse"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
input_dir = Path(PATH_INPUT) / name_source
valid_ext = {".fasta", ".fa", ".faa", ".txt"}
dfs = []

for file in input_dir.iterdir():
    if file.is_file() and file.suffix.lower() in valid_ext:
        df = read_fasta_doc(file)
        dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

In [4]:
df["label"] = pd.NA 

df.loc[df["id"].str.contains("Positive", case=False, na=False), "label"] = 1
df.loc[df["id"].str.contains("Negative", case=False, na=False), "label"] = 0

df = df[["sequence", "label"]]
df.shape

(11036, 2)

- Checking duplicates

In [5]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [6]:
df_full.shape

(11036, 2)

In [7]:
df_errors.shape

(0, 1)

- Working with metada

In [8]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [9]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences" : len(df_errors),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'Apache 2.0',
 'year of publication': 2025,
 'last update date': datetime.datetime(2025, 7, 24, 0, 0),
 'download date': Timestamp('2025-10-17 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'toxic',
 'dataset information': 'Negative;Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from uniprot, Sampling from another DB',
 'repository or server': 'https://github.com/cbbl-skku-org/HyPepTox-Fuse/tree/main/raw_dataset',
 'publication': 'https://www.sciencedirect.com/science/article/pii/S2095177925002278',
 'number_of_raw_sequences': 11036,
 'number_of_sequences_retained': 11036,
 'number_of_positive_sequences': 5518,
 'number_of_negative_sequences': 5518,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [10]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [11]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)